# MySpeech-to-MySpeech
Google Colab project that takes what you say and makes it sound unbearably sophisticated.

**Pipeline:**
1. Transcribe a voice message with Whisper
2. Remove filler words (ähm, äh)
3. Rewrite in pompous academic German using Claude
4. Synthesize back in your own voice using XTTS v2

**Requirements:**
- Colab GPU runtime (T4 is sufficient)
- Anthropic API key stored in Colab Secrets as `claude-key`
- A voice message: `message.wav`
- 1–3 short voice samples of yourself: `*.wav` (~10–30 sec each)

## 1. Installation

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install python3.11 python3.11-venv python3.11-dev ffmpeg -y -qq

!python3.11 -m venv /content/venv
!/content/venv/bin/pip install --upgrade pip -q

!/content/venv/bin/pip install openai-whisper -q
!/content/venv/bin/pip install TTS -q
!/content/venv/bin/pip install anthropic -q
!/content/venv/bin/pip install pydub -q
!/content/venv/bin/pip install transformers==4.40.0 -q
!/content/venv/bin/pip install torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118 -q
!/content/venv/bin/pip install accelerate -q


## 2. Imports

In [ ]:
import os
import re
import glob

# Fix matplotlib backend before importing TTS
if "MPLBACKEND" in os.environ:
    del os.environ["MPLBACKEND"]
import matplotlib
matplotlib.use("agg")

import whisper
from anthropic import Anthropic
from TTS.api import TTS
from pydub import AudioSegment
from google.colab import files, userdata

## 3. Upload Audio Files
Upload your voice message (`message.wav`) and 1–3 voice samples for cloning.

In [ ]:
print("Upload: message.wav + voice samples (e.g. voice_sample_1.wav)")
uploaded = files.upload()
print(f"✓ Uploaded: {list(uploaded.keys())}")

## 4. Transcription & Filler Word Removal
Whisper transcribes the voice message.

In [ ]:
# Transcribe
print("Transcribing...")
model = whisper.load_model("medium")
result = model.transcribe("message.wav", language="de")
original = result["text"]
print(f"Original:\n{original}\n")

# Save for next steps
with open("clean_text.txt", "w") as f:
    f.write(original)

## 5. Text Elevation via Claude
Claude rewrites the cleaned text in an exaggeratedly pompous, 19th-century academic style.
You can change the prompt here for different transformations.

In [ ]:
# Load API key from Colab Secrets
api_key = userdata.get("claude-key")
client = Anthropic(api_key=api_key)

# Load cleaned text
with open("clean_text.txt", "r") as f:
    text = f.read()
    
# Rewrite text with Claude
message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": f"""Formuliere diesen Text in extrem hochgestochener, prätentiöser deutscher Sprache um.
Nutze komplizierte Fremdwörter, verschachtelte Sätze und einen übertrieben akademischen Stil.
Es soll klingen wie ein Gelehrter aus dem 19. Jahrhundert.
Behalte die BEDEUTUNG bei - verändere nicht den Inhalt oder die Aussage.
Antworte NUR mit dem umformulierten Text, keine Erklärungen, kein Kommentar.

{text}"""}
    ]
)

elevated_text = message.content[0].text
print(f"Elevated:\n{elevated_text}")

## 6. Voice Synthesis
XTTS v2 clones your voice from the uploaded samples and synthesizes the elevated text.
The text is split into sentences first to avoid XTTS character limits, then merged with pydub.

In [ ]:
# Find uploaded voice samples
voice_samples = [f for f in os.listdir(".") if f.endswith(".wav") and f != "message.wav"]
print(f"Using voice samples: {voice_samples}")

# Split into sentences to avoid XTTS character limit
sentences = re.split(r"(?<=[.!?])\s+", elevated_text)
sentences = [s.strip() for s in sentences if s.strip()]

# Load TTS model
print("Loading XTTS model...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")

# Synthesize each sentence
audio_parts = []
for i, sentence in enumerate(sentences):
    print(f"  Sentence {i+1}/{len(sentences)}...")
    tts.tts_to_file(
        text=sentence,
        speaker_wav=voice_samples,
        language="de",  #change if your output is in another language 
        file_path=f"part_{i}.wav"
    )
    audio_parts.append(AudioSegment.from_wav(f"part_{i}.wav"))

# Merge and export
combined = audio_parts[0]
for part in audio_parts[1:]:
    combined += part
combined.export("output.wav", format="wav")

# Cleanup temp files
for f in glob.glob("part_*.wav"):
    os.remove(f)

print("✓ Audio generated: output.wav")

## 7. Download Result

In [ ]:
files.download("output.wav")